In [ ]:
!pip install -q streamlit openai pandas numpy
!npm install -g localtunnel

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
from openai import OpenAI

# Page config
st.set_page_config(page_title="File Q&A App", layout="wide")

st.title("📄 Ask Questions About Your File")

# API key input
api_key = st.text_input("Enter your OpenAI API Key", type="password")

if api_key:
    client = OpenAI(api_key=api_key)

    uploaded_file = st.file_uploader("Upload CSV or TXT file", type=["csv", "txt"])

    if uploaded_file is not None:

        # Read file
        if uploaded_file.name.endswith(".csv"):
            df = pd.read_csv(uploaded_file)
            st.subheader("Data Preview")
            st.dataframe(df.head())

            context = df.to_string()

        else:
            context = uploaded_file.read().decode("utf-8")
            st.text_area("File Content", context, height=200)

        # Ask question
        question = st.text_input("Ask a question about your file")

        if question:
            with st.spinner("Thinking..."):

                prompt = f"""
                You are a helpful assistant.
                Answer the question based on the data below.

                Data:
                {context[:3000]}

                Question:
                {question}
                """

                response = client.chat.completions.create(
                    model="gpt-4.1-mini",
                    messages=[
                        {"role": "user", "content": prompt}
                    ]
                )

                answer = response.choices[0].message.content

                st.subheader("Answer")
                st.success(answer)

else:
    st.warning("Please enter your OpenAI API key")

In [ ]:
!pip freeze > requirements.txt

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!npx localtunnel --port 8501